In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="True"

from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
import glob
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

docs_dir= "docs_dir" #jaruri ni ki is folder ko niche dene ke liye variable me name save karna pade ham simplification ke liye folder ka name as a string bhi de sakte h 
index_dir= "faiss_index" #isko fxn me use ni kiya niche vector databse banate me use kiya h and isko bhi as a string de sakte h, no need to save in variable 

def load_and_split(docs_dir):
    chunks=[] #chunks me ham har file ko open karke uske para ko split and strip karke fir har para ko chance by chance loop ke help se append karenge and metadata me para kis file ka h isko mention karenge 
    for path in glob.glob(os.path.join(docs_dir,"*.txt")): #file ko iterate karne ke liye hamesa glob use kiya jata h (glob.glob pattern se match karti file dhoondkar list deta h)
        #os.path se path or .join se folder path aur file pattern ko sahi tarah jodta h 
        with open(path, "r", encoding="utf-8") as f:
            text= f.read() #text me file read karke 
        source= os.path.basename(path) #source me file ka basename lete h (yha pe path variable iteration me chal rha file ka name h)
        paragraph= [p.strip() for p in text.split("\n\n") if p.strip()] #ye \n\n matlab para ending pe line ke basis pe split karke uske starting ya ending me jo whitespace,tabes or blank-lines ko hata deta h (edge case ki para ke baad 2 line ho to bo ek empty para ke rup me aayega jiska strip nikalne se empty para add hoga isliye if p.strip likha h jisse agar strip hone ke baad kuch bachta ho tabhi bo paragraph me add ho)
        current=""
        for para in paragraph:
            if current and len(current)+len(para)+2 > 500:
                chunks.append(Document(page_content=current, metadata={"source":source}))
                current=para
            else:
                current= (current + "\n\n" + para).strip() #pehli baar me current me para jodta h and starting me line bhi isliye strip use kiya then agli baar agar if condition me 500 wala satisfied ni hota to existing para me new para jodta h with one line gap then jab tak jodta h tab tak 500 satisfied ni ho 
                #usi ke baad append karke (with metadata about source file) dobara already chal rahe para se shuru karta h. (concept of chunking pichle chunk ka kuch hissa agle chunk me for better results)
                #but with strip jisse starting or ending me whitespace na aaye (vese 1st case ke baad esa koi case ni hota)
        if current: #agar last ka para 500 se chota hua to is case me loop end hoga par current me fir bhi value hogi and us para/current ko chunks me ye if save karega
            chunks.append(Document(page_content=current, metadata={"source":source}))
    return chunks

chunks=load_and_split(docs_dir)
print(f"{len(chunks)} chunks")

6 chunks


In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(index_dir) #jo variable upar define kiya tha 
print(f"Index saved to {index_dir}/")

Index saved to faiss_index/


In [37]:
print("\\")

\
